# 08 - Fine-tune ArTST v3 (Audio → Diacritized Text)

**Base Model:** `MBZUAI/artst_asr_v3`
**Architecture:** SpeechT5 (151M params)
**Key Feature:** Outputs diacritized Arabic text directly from audio!

## Why ArTST v3?
- **End-to-end**: Audio → Diacritized text in one model
- **Arabic-native**: Trained specifically on Arabic speech
- **Efficient**: 151M params (smaller than Whisper)

## Training Techniques:
1. **Audio Augmentation**: Speed perturbation, noise injection
2. **Gradient Checkpointing**: Memory efficient training
3. **Mixed Precision**: FP16 for faster training

## Tasks:
1. Load KSAA audio + text data
2. Fine-tune ArTST v3
3. Evaluate on dev set
4. Generate test submission

In [22]:
!pip install -q transformers torch accelerate datasets tqdm librosa soundfile pandas

In [23]:
# Setup
import os, sys, json, re, torch, gc, zipfile
from datetime import datetime
from tqdm import tqdm
from pathlib import Path
import pandas as pd
import librosa
import numpy as np
import warnings
warnings.filterwarnings('ignore')

IS_RUNPOD = False  # Cloud detection removed
PROJECT_DIR = Path('.')
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))

# Import utilities (includes safeguards for empty responses & separated characters)
from utils_diacritization import (
    normalize_arabic, postprocess_diacritics, compute_metrics,
    is_valid_output, repair_output, get_safe_seq2seq_generation_config
)

DATA_DIR = PROJECT_DIR / 'Public_Data_TrainDev'
TRAIN_DIR = DATA_DIR / 'Train'
TRAIN_AUDIO_DIR = TRAIN_DIR / 'train_audio'
TRAIN_TEXT_DIR = TRAIN_DIR / 'train_text'
DEV_INPUT = DATA_DIR / 'dev input-output' / 'Dev_input.json'
DEV_OUTPUT = DATA_DIR / 'dev input-output' / 'Dev_output.json'
DEV_AUDIO_DIR = DATA_DIR / 'Dev' / 'dev_audio'
TEST_DIR = PROJECT_DIR / 'Test'
TEST_INPUT = TEST_DIR / 'test_input.json'
TEST_AUDIO_DIR = TEST_DIR / 'test_audio'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
SUBMISSION_DIR = PROJECT_DIR / 'submissions'
CHECKPOINTS_DIR = PROJECT_DIR / 'checkpoints' / 'artst_v3_ft'

for d in [OUTPUT_DIR, SUBMISSION_DIR, CHECKPOINTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Environment: 'Local' | Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")

def clear_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

Environment: Local | Device: cuda
GPU: NVIDIA RTX A5000 (23.6 GB)


## 1. Load Training Data (Audio + Text Pairs)

In [24]:
train_tsv = TRAIN_DIR / 'train_list.tsv'
train_df = pd.read_csv(train_tsv, sep='\t')
print(f"Training samples: {len(train_df)}")

def load_audio(path, sr=16000):
    """Load audio file and resample to 16kHz."""
    try:
        audio, _ = librosa.load(path, sr=sr)
        return audio
    except Exception as e:
        return None

def load_training_data(max_samples=None):
    """Load audio-text pairs for training."""
    data = []
    
    for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Loading data"):
        if max_samples and len(data) >= max_samples:
            break
            
        stem = row['stem']
        audio_path = TRAIN_AUDIO_DIR / f"{stem}.mp3"
        text_path = TRAIN_TEXT_DIR / f"{stem}.txt"
        
        if audio_path.exists() and text_path.exists():
            with open(text_path, 'r', encoding='utf-8') as f:
                diacritized = normalize_arabic(f.read().strip())
            
            data.append({
                'id': stem,
                'audio_path': str(audio_path),
                'text_diacritized': diacritized
            })
    
    return data

# Load all training data (or subset for testing)
train_data = load_training_data(max_samples=None)  # Set to 500 for quick test
print(f"Loaded {len(train_data)} audio-text pairs")

Training samples: 2327


Loading data: 100%|██████████| 2327/2327 [00:11<00:00, 198.13it/s]

Loaded 2327 audio-text pairs


## 2. Audio Augmentation

In [25]:
import random

def speed_perturbation(audio, sr=16000, speed_factor=None):
    """Apply speed perturbation (0.9x - 1.1x)."""
    if speed_factor is None:
        speed_factor = random.uniform(0.9, 1.1)
    return librosa.effects.time_stretch(audio, rate=speed_factor)

def add_noise(audio, noise_level=0.005):
    """Add random Gaussian noise."""
    noise = np.random.randn(len(audio)) * noise_level
    return audio + noise

def augment_audio(audio, augment_prob=0.3):
    """Apply random augmentation to audio."""
    if random.random() > augment_prob:
        return audio
    
    aug_type = random.choice(['speed', 'noise', 'both'])
    
    if aug_type in ['speed', 'both']:
        audio = speed_perturbation(audio)
    if aug_type in ['noise', 'both']:
        audio = add_noise(audio, noise_level=random.uniform(0.002, 0.01))
    
    return audio

print("Audio augmentation functions ready")

Audio augmentation functions ready


## 3. Load Model

In [26]:
from transformers import SpeechT5Processor, SpeechT5ForSpeechToText

MODEL_NAME = 'MBZUAI/artst_asr_v3'
MODEL_KEY = 'artst_v3_ft'

print(f"Loading {MODEL_NAME}...")
processor = SpeechT5Processor.from_pretrained(MODEL_NAME)
model = SpeechT5ForSpeechToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,  # Full precision for training
).to(device)

model.gradient_checkpointing_enable()

print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")
clear_gpu()

Loading MBZUAI/artst_asr_v3...


Loading weights:   0%|          | 0/369 [00:00<?, ?it/s]

SpeechT5ForSpeechToText LOAD REPORT from: MBZUAI/artst_asr_v3
Key                                                  | Status     |  | 
-----------------------------------------------------+------------+--+-
speecht5.encoder.prenet.pos_sinusoidal_embed.weights | UNEXPECTED |  | 
speecht5.decoder.prenet.embed_positions.weights      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: 151,258,752 parameters


## 4. Prepare Dataset

In [27]:
from datasets import Dataset
from dataclasses import dataclass
from typing import Any, Dict, List, Union

MAX_AUDIO_LENGTH = 16000 * 30  # 30 seconds max
MAX_TEXT_LENGTH = 256

def preprocess_function(examples):
    """Process audio and text for training."""
    audio_arrays = []
    labels = []
    
    for audio_path, text in zip(examples['audio_path'], examples['text_diacritized']):
        # Load and optionally augment audio
        audio = load_audio(audio_path)
        if audio is None:
            audio = np.zeros(16000)  # 1 second of silence as fallback
        else:
            audio = augment_audio(audio, augment_prob=0.3)
        
        # Truncate long audio
        if len(audio) > MAX_AUDIO_LENGTH:
            audio = audio[:MAX_AUDIO_LENGTH]
        
        audio_arrays.append(audio)
        labels.append(text)
    
    # Process audio
    inputs = processor(
        audio=audio_arrays,
        sampling_rate=16000,
        return_tensors="np",
        padding=True
    )
    
    # Process text labels
    label_ids = processor.tokenizer(
        labels,
        max_length=MAX_TEXT_LENGTH,
        truncation=True,
        padding='max_length'
    )
    
    return {
        'input_values': inputs['input_values'].tolist(),
        'labels': label_ids['input_ids']
    }

# Create dataset (use subset if needed)
train_subset = train_data  # Use all data or train_data[:500] for testing

train_dataset = Dataset.from_list(train_subset)
train_dataset = train_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=8,
    remove_columns=['id', 'audio_path', 'text_diacritized'],
    desc="Processing audio"
)

print(f"Train dataset: {len(train_dataset)} samples")

Processing audio:   0%|          | 0/2327 [00:00<?, ? examples/s]

[src/libmpg123/id3.c:process_comment():587] error: No comment text / valid description?


Train dataset: 2327 samples


In [28]:
# Custom data collator for speech
@dataclass
class DataCollatorSpeechSeq2Seq:
    processor: Any
    
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Pad input values
        input_features = [torch.tensor(f['input_values']) for f in features]
        input_features = torch.nn.utils.rnn.pad_sequence(
            input_features, batch_first=True, padding_value=0
        )
        
        # Pad labels
        labels = [torch.tensor(f['labels']) for f in features]
        labels = torch.nn.utils.rnn.pad_sequence(
            labels, batch_first=True, padding_value=-100
        )
        
        return {
            'input_values': input_features,
            'labels': labels
        }

data_collator = DataCollatorSpeechSeq2Seq(processor)

## 5. Training

In [29]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# Training args - Optimized for 24GB GPU (audio uses more memory)
training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINTS_DIR),
    
    # Training - Audio uses more memory, but model is small (151M)
    num_train_epochs=5,
    per_device_train_batch_size=4,   # Audio batches need more memory
    gradient_accumulation_steps=4,   # Effective batch: 16
    
    # Optimizer
    learning_rate=5e-6,  # Very low for fine-tuning speech model
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    
    # Precision - FP16 for speed (audio models handle FP16 well)
    fp16=True,
    
    # Saving
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    
    # Logging
    logging_steps=50,
    report_to="none",
    
    # Generation
    predict_with_generate=True,
    generation_max_length=MAX_TEXT_LENGTH,
    
    # Performance
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print("Trainer configured")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accum: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  FP16: {training_args.fp16}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer configured
  Epochs: 5
  Batch size: 4
  Gradient accum: 4
  Effective batch: 16
  FP16: True


In [30]:
# Train
clear_gpu()

checkpoint = None
checkpoints = list(CHECKPOINTS_DIR.glob('checkpoint-*'))
if checkpoints:
    checkpoint = str(max(checkpoints, key=lambda x: int(x.name.split('-')[1])))
    print(f"Resuming from: {checkpoint}")

print("Starting training...")
trainer.train(resume_from_checkpoint=checkpoint)

Resuming from: ./checkpoints/artst_v3_ft/checkpoint-730
Starting training...


There were missing keys in the checkpoint model loaded: ['text_decoder_postnet.lm_head.weight'].


Step,Training Loss


TrainOutput(global_step=730, training_loss=0.0, metrics={'train_runtime': 0.0034, 'train_samples_per_second': 3441275.442, 'train_steps_per_second': 215911.566, 'total_flos': 2.1851966909645983e+18, 'train_loss': 0.0, 'epoch': 5.0})

In [31]:
# Save final model
final_path = CHECKPOINTS_DIR / 'final'
trainer.save_model(str(final_path))
processor.save_pretrained(str(final_path))
print(f"Saved to: {final_path}")
clear_gpu()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: ./checkpoints/artst_v3_ft/final


## 6. Inference Functions

In [32]:
# Load fine-tuned model for inference (FP32 for consistency with training)
final_model_path = CHECKPOINTS_DIR / 'final'
if final_model_path.exists():
    print(f"Loading fine-tuned model from {final_model_path}")
    # Clear training model first
    del model
    clear_gpu()
    
    model = SpeechT5ForSpeechToText.from_pretrained(
        final_model_path,
        torch_dtype=torch.float32  # FP32 to match input tensors
    ).to(device)
    processor = SpeechT5Processor.from_pretrained(final_model_path)
else:
    print("Using model from training")

model.eval()
print("Model ready for inference (FP32)")

Loading fine-tuned model from ./checkpoints/artst_v3_ft/final


Loading weights:   0%|          | 0/369 [00:00<?, ?it/s]

Model ready for inference (FP32)


In [33]:
@torch.inference_mode()
def transcribe_diacritized(audio_path: str, fallback_text: str = "") -> str:
    """
    Transcribe audio directly to diacritized Arabic text.
    Includes safeguards against empty responses.
    """
    audio = load_audio(audio_path)
    if audio is None:
        return fallback_text
    
    # Truncate if too long
    if len(audio) > MAX_AUDIO_LENGTH:
        audio = audio[:MAX_AUDIO_LENGTH]
    
    inputs = processor(audio=audio, sampling_rate=16000, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Get safe generation config
    gen_config = get_safe_seq2seq_generation_config()
    gen_config['max_length'] = MAX_TEXT_LENGTH
    
    generated_ids = model.generate(**inputs, **gen_config)
    transcription = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    # Repair output (validates and fixes issues)
    transcription = repair_output(transcription.strip(), fallback_text, fallback_to_input=True)
    
    return transcription

## 7. Evaluate on Dev Set

In [34]:
with open(DEV_INPUT, 'r', encoding='utf-8') as f:
    dev_input = json.load(f)
with open(DEV_OUTPUT, 'r', encoding='utf-8') as f:
    dev_output = json.load(f)

CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_checkpoint.json"

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_checkpoint(predictions):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({'processed_ids': [p['id'] for p in predictions], 'predictions': predictions}, f, ensure_ascii=False)

processed_ids, dev_predictions = load_checkpoint()
print(f"Dev: {len(dev_input)} samples, {len(processed_ids)} already done")

for item in tqdm(dev_input, desc="Dev Set"):
    if item['id'] in processed_ids:
        continue
    try:
        audio_path = DEV_AUDIO_DIR / f"{item['id']}.mp3"
        result = transcribe_diacritized(str(audio_path), item['text_undiacritized'])
        dev_predictions.append({'id': item['id'], 'text_diacritized': result})
        save_checkpoint(dev_predictions)
    except Exception as e:
        print(f"Error: {e}")
        dev_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_checkpoint(dev_predictions)

with open(OUTPUT_DIR / f"{MODEL_KEY}_dev_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(dev_predictions, f, ensure_ascii=False, indent=2)

Dev: 260 samples, 260 already done


Dev Set: 100%|██████████| 260/260 [00:00<00:00, 1608435.16it/s]


In [35]:
# Compute metrics
print("\n" + "="*60)
print("DEV SET RESULTS")
print("="*60)

metrics = compute_metrics(dev_predictions, dev_output, exclude_irab=False)
print(f"\n[Including I'rab]")
print(f"  DER: {metrics['DER']*100:.2f}%")
print(f"  WER: {metrics['WER']*100:.2f}% (PRIMARY)")
print(f"  SER: {metrics['SER']*100:.2f}%")


DEV SET RESULTS

[Including I'rab]
  DER: 78.76%
  WER: 97.48% (PRIMARY)
  SER: 99.62%


## 8. Generate Test Submission

In [36]:
with open(TEST_INPUT, 'r', encoding='utf-8') as f:
    test_input = json.load(f)

TEST_CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_test_checkpoint.json"

def load_test_checkpoint():
    if TEST_CHECKPOINT_FILE.exists():
        with open(TEST_CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_test_checkpoint(predictions):
    with open(TEST_CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({'processed_ids': [p['id'] for p in predictions], 'predictions': predictions}, f, ensure_ascii=False)

test_processed_ids, test_predictions = load_test_checkpoint()
print(f"Test: {len(test_input)} samples, {len(test_processed_ids)} already done")

for item in tqdm(test_input, desc="Test Set"):
    if item['id'] in test_processed_ids:
        continue
    try:
        audio_path = TEST_AUDIO_DIR / f"{item['id']}.mp3"
        result = transcribe_diacritized(str(audio_path), item['text_undiacritized'])
        test_predictions.append({'id': item['id'], 'text_diacritized': result})
        save_test_checkpoint(test_predictions)
    except:
        test_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_test_checkpoint(test_predictions)

with open(OUTPUT_DIR / f"{MODEL_KEY}_test_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

Test: 328 samples, 328 already done


Test Set: 100%|██████████| 328/328 [00:00<00:00, 1492116.82it/s]


In [37]:
# Create submission
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

json_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission.json"
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

zip_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission_{timestamp}.zip"
with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(json_file, 'submission.json')

print("="*60)
print("SUBMISSION READY")
print("="*60)
print(f"ZIP: {zip_file}")

SUBMISSION READY
ZIP: ./submissions/artst_v3_ft_submission_20260224_081956.zip


In [38]:
# Sync and cleanup
import subprocess
sync_script = PROJECT_DIR / 'sync_results.sh'
if sync_script.exists() and False:
    print("Syncing to Google Drive...")
    subprocess.run(['bash', str(sync_script)])

# Final cleanup - free GPU memory
print("\nCleaning up GPU memory...")
del model
del processor
clear_gpu()
print("Done! GPU memory released.")

Syncing to Google Drive...
KSAA 2026 - Sync Results to Google Drive
  Project: .

[1/3] Syncing outputs...
Transferred:   	          0 B / 9.066 KiB, 0%, 0 B/s, ETA -
Checks:                37 / 39, 95%
Transferred:            0 / 1, 0%
Elapsed time:         0.8s
Checking:
 *              artst_v3_ft_dev_predictions.json: checking
 *             artst_v3_ft_test_predictions.json: checking
Transferring:
 *           whisper_tashkeel_ft_checkpoint.json:  0% /9.066Ki, 0/s, -Transferred:   	    9.066 KiB / 9.066 KiB, 100%, 0 B/s, ETA -
Checks:                42 / 42, 100%
Transferred:            0 / 1, 0%
Elapsed time:         1.3s
Transferring:
 *           whisper_tashkeel_ft_checkpoint.json:100% /9.066Ki, 0/s, -Transferred:   	    9.066 KiB / 9.066 KiB, 100%, 9.063 KiB/s, ETA 0s
Checks:                42 / 42, 100%
Transferred:            0 / 1, 0%
Elapsed time:         1.8s
Transferring:
 *           whisper_tashkeel_ft_checkpoint.json:100% /9.066Ki, 9.064Ki/s, 0sTransferred:   	    9.